# 미니 힌트 실험 조종석 (VISION 트랙 2)

**목적**: 구조 정보 주입의 2×2 효과 분리 측정 — `PLAN_post_labeling.md` 게이트制.

| | 힌트 없음 | CLIP 유사쌍 힌트 |
|---|---|---|
| 직답 | mini_v1_aug2 = **15.5%** (기준, 완료) | **mini_hint_aug2** (v6_hint) |
| CoT (gemma events) | **mini_gemma_cot_aug2** (v7_cot) | mini_cot_hint_aug2 (v7_cot_hint, 승자 확인 후) |

**게이트**: vs 15.5%, **+4%p↑만 확전** (미니-full 괴리 전례: exp14). 미니 = 1000샘플·aug2·300스텝 (~1.5h+평가).

**주의**
- GPU 빈 슬롯에서만 (⓪에서 라벨링/학습 점유 확인). 미니는 gemma 라벨링과 동시 실행 금지
- 힌트 쌍 번호는 증강 변형마다 재매핑됨 — ②의 정합성 검사가 통과해야 학습 의미 있음
- CoT 평가는 --max-new-tokens 512 (레지스트리에 내장, 큐가 자동 적용)

In [ ]:
# ⓪ 상태 보드 — GPU·라벨링·학습 점유 확인
import subprocess, glob, os
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
prog = "./outputs/gemma_labels/progress.txt"
if os.path.exists(prog):
    print("라벨링:", open(prog, encoding="utf-8").read().strip(), "<- 돌고 있으면 미니 시작 금지")
out = subprocess.run(["powershell", "-NoProfile", "-Command",
                      "@(Get-CimInstance Win32_Process -Filter \"Name='python.exe'\" "
                      "| Where-Object { $_.CommandLine -match 'train(_cot)?\\.py|gemma_label' }).Count"],
                     capture_output=True, text=True)
print(f"학습/라벨링 프로세스: {out.stdout.strip()}개 (0이어야 시작 가능)")

In [ ]:
# ① 재료 검증 — CLIP 유사쌍 분포 + gemma 미니 풀 커버리지 + 힌트 미리보기
import sys; sys.path.insert(0, "scripts")
import pandas as pd
from structure_features import load_clip_pairs, load_gemma_labels, hint_text
from train import load_excluded_ids

pairs = load_clip_pairs()
n_sim = pd.Series({k: len(v) for k, v in pairs.items()})
print("유사쌍 개수 분포 (train 전량):")
print(n_sim.value_counts().sort_index().to_string())

train_df = pd.read_csv("snuaichallenge_data/train.csv")
train_df = train_df[~train_df.Id.isin(load_excluded_ids())].reset_index(drop=True)
mini_pool = train_df.sample(n=1000, random_state=42)          # train.py --max-samples 1000과 동일
g = load_gemma_labels()
cover = mini_pool.Id.isin(set(g.Id)).mean()
print(f"\ngemma 라벨의 미니 풀 커버리지: {cover:.1%}  <- 100%여야 CoT 미니 가능")

print("\n힌트 미리보기 (원본 제시 순서):")
for sid in mini_pool.Id.head(5):
    print(f"  [{sid}] {hint_text(pairs.get(sid, [])).strip()}")

In [ ]:
# ② 재매핑 정합성 자동 검사 — 증강 변형의 힌트 쌍 번호가 맞는지 (통과해야 진행)
import random
from structure_features import PAIR_COLS, load_clip_features, remap_pairs

clip = load_clip_features().set_index("Id")
rng = random.Random(0)
n_ok = 0
ids = list(pairs)[:200]
for sid in ids:
    perm = [0, 1, 2, 3]; rng.shuffle(perm)
    a = set(remap_pairs(pairs[sid], perm))
    b = set()
    for i in range(1, 5):
        for j in range(i + 1, 5):
            o1, o2 = perm[i - 1] + 1, perm[j - 1] + 1
            if clip.loc[sid][PAIR_COLS[(min(o1, o2), max(o1, o2))]] < 0.20:
                b.add((i, j))
    n_ok += (a == b)
print(f"재매핑 교차 검증: {n_ok}/{len(ids)} 일치 {'- 통과' if n_ok == len(ids) else '!! 불일치 - 진행 금지'}")

In [ ]:
# ③ 실험 레지스트리 — 미니 3종 + 커맨드 생성
MINI = "--max-samples 1000 --max-steps 300"
DEFAULTS = dict(model="./models/Qwen3-VL-2B-Instruct", aug_mult=2, lr=1e-4, epochs=1,
                max_hours=10, script="scripts/train.py", extra="", eval_tokens=None)

EXPERIMENTS = {
    "mini_hint_aug2":      dict(prompt="v6_hint", extra=MINI),
    "mini_gemma_cot_aug2": dict(prompt="v7_cot", script="scripts/train_cot.py",
                                extra=MINI + " --events-from gemma", eval_tokens=512),
    "mini_cot_hint_aug2":  dict(prompt="v7_cot_hint", script="scripts/train_cot.py",
                                extra=MINI + " --events-from gemma", eval_tokens=512),
}

def cfg(name): return {**DEFAULTS, **EXPERIMENTS[name]}

def train_cmd(name, smoke=False):
    c = cfg(name)
    run = name + ("_smoke" if smoke else "")
    cmd = (f"python {c['script']} --run-name {run} --model {c['model']} "
           f"--aug-mult {c['aug_mult']} --lr {c['lr']} --epochs {c['epochs']} "
           f"--max-hours {c['max_hours']} --prompt {c['prompt']} {c['extra']}").strip()
    if smoke: cmd += " --max-samples 12 --grad-accum 4 --max-steps 5"   # argparse 뒤가 우선
    return cmd

def eval_cmd(name):
    c = cfg(name)
    cmd = (f"python scripts/eval_zero_shot.py --model {c['model']} "
           f"--adapter ./outputs/runs/{name}/adapter --prompt {c['prompt']}")
    if c["eval_tokens"]: cmd += f" --max-new-tokens {c['eval_tokens']}"
    return cmd

def queue_entry(name):
    """train_queue.py --queue 항목 ("이름|인자") — --script/--eval-max-new-tokens 의사 플래그 포함."""
    c = cfg(name)
    e = (f"--script {os.path.basename(c['script'])} --model {c['model']} "
         f"--aug-mult {c['aug_mult']} --lr {c['lr']} --epochs {c['epochs']} "
         f"--max-hours {c['max_hours']} --prompt {c['prompt']} {c['extra']}")
    if c["eval_tokens"]: e += f" --eval-max-new-tokens {c['eval_tokens']}"
    return f"{name}|{e.strip()}"

for n in EXPERIMENTS: print(f"[{n}]\n  {train_cmd(n)}\n  {eval_cmd(n)}")

In [ ]:
# ④ CoT 타깃 미리보기 — gemma events 형식 눈검사 (GPU 불필요)
!python scripts/train_cot.py --run-name preview --preview 3 --events-from gemma --prompt v7_cot_hint

In [ ]:
# ⑤ 스모크 (~3분/개) — 코드 변경 후 첫 실행이면 필수. peak VRAM 7.5GB 초과 시 중단
NAME = "mini_hint_aug2"
!{train_cmd(NAME, smoke=True)}

In [ ]:
# ⑥ 미니 체인 실행 — train_queue.py 백그라운드 (학습→평가 자동, 커널 죽어도 계속)
import subprocess, sys, os
CHAIN = ["mini_hint_aug2", "mini_gemma_cot_aug2"]   # 조합(mini_cot_hint_aug2)은 승자 확인 후 추가
qargs = []
for n in CHAIN: qargs += ["--queue", queue_entry(n)]
proc = subprocess.Popen([sys.executable, "scripts/train_queue.py"] + qargs,
                        stdout=open("outputs/mini_chain_console.log", "w", encoding="utf-8"),
                        stderr=subprocess.STDOUT, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(f"체인 시작: PID {proc.pid} | 예약 {CHAIN}")
print("진행: outputs/train_queue.log | 개별 콘솔: outputs/runs/<이름>/console.log")

In [ ]:
# ⑦ 큐 로그 테일 — 30초 갱신, '체인 큐 종료' 찍히면 자동 중단
import time
from IPython.display import clear_output
while True:
    try:
        lines = open("outputs/train_queue.log", encoding="utf-8").read().splitlines()
    except OSError:
        lines = ["(로그 대기 중)"]
    clear_output(wait=True)
    print("\n".join(lines[-12:]))
    if lines and "체인 큐 종료" in lines[-1]:
        break
    time.sleep(30)

In [ ]:
# ⑧ 결과표 + 게이트 판정 + 세그먼트 분해
import glob
import pandas as pd
from structure_features import camera_regex

BASE_NAME, BASE, NEED = "mini_v1_aug2", 0.1548, 0.04
exp = pd.read_csv("./outputs/experiments.csv")
print(f"기준: {BASE_NAME} {BASE:.1%} | 게이트 +{NEED:.0%}p\n")
for name in ["mini_hint_aug2", "mini_gemma_cot_aug2", "mini_cot_hint_aug2"]:
    rows = exp[exp.model_id.str.contains(name, na=False)]
    if not len(rows):
        continue
    acc = rows.iloc[-1].acc_shuffled
    diff = acc - BASE
    verdict = "승 - 확전 후보" if diff >= NEED else ("보류(노이즈)" if diff > -NEED else "기각")
    print(f"{name:24s} {acc:.1%} ({diff:+.1%}) -> {verdict}")

    preds = sorted(glob.glob(f"./outputs/preds/*{name}*.csv"))
    if preds:
        p = pd.read_csv(preds[-1])
        p = p[p["gt"] != "[1, 2, 3, 4]"]
        p["camera_re"] = p.Sentence.map(camera_regex)
        p["has_pair"] = p.Id.map(lambda i: len(pairs.get(i, [])) > 0)
        for col, lab in [("camera_re", "카메라"), ("has_pair", "유사쌍")]:
            t = p.groupby(col).correct.agg(["mean", "size"])
            print("   " + " | ".join(f"{lab}{'O' if k else 'X'} {v['mean']:.1%}(n={int(v['size'])})"
                                     for k, v in t.iterrows()))

In [ ]:
# ⑨ 확전 가이드 — 승자의 full 학습 커맨드 (Structure_Pipeline 레지스트리용)
# 판정 규칙 (PLAN_post_labeling.md):
#   hint만 승  -> v6_hint aug2 full (~20h)          | cot만 승 -> v7_cot full (분해 품질 가설 증명)
#   둘 다 승   -> mini_cot_hint_aug2 추가 후 최강 조합 full | 둘 다 패 -> 트랙 2 종료, 증강 축·4B로
WINNER = None   # 예: "mini_hint_aug2"
if WINNER:
    c = cfg(WINNER)
    full = (f"python {c['script']} --run-name {WINNER.replace('mini_', 'exp16_')}_full "
            f"--model {c['model']} --aug-mult 2 --lr {c['lr']} --epochs 1 --max-hours 0 "
            f"--prompt {c['prompt']} --snapshot-steps 150"
            + (" --events-from gemma" if "cot" in WINNER else ""))
    print(full)
else:
    print("⑧ 판정 후 WINNER 지정")

---
## ⑩ 타깃 증강 설정 (트랙 1 확장 — full 학습용 가중 CSV 생성)

조건·배수의 **숫자만 바꿔 재실행**하면 `train.py --aug-weights`용 CSV가 갱신된다.

⚠️ **규정 (PROJECT_SETUP §4.3)**: "평가 데이터 특성 분석 후 학습" = **실격**.
조건 선정의 근거는 **holdout 약점 실측**으로만 문서화할 것 (test 분포 인용 금지).
현재 검증된 약점 축: 카메라X 35.6% / scene_cuts=0 26.8% / 사건 1~2개 16~24% (모두 holdout×exp07).

In [ ]:
# ⑩ 증강 규칙 편집 — (설명, 조건, 배수) 위에서부터 첫 매치, 미매치는 DEFAULT_MULT
from structure_features import build_feature_table, make_aug_weights

RULES = [
    # 축: r.camera_re(정규식) / r.scene_cuts(CLIP) / r.n_similar(유사쌍수) /
    #     r.has_gemma and r.n_events, r.n_subj, r.n_markers (gemma, 커버리지 확인)
    ("카메라X + 전환없음(최약점 교차)", lambda r: (not r.camera_re) and r.scene_cuts == 0, 5),
    ("카메라X",                        lambda r: not r.camera_re,                         4),
    ("전환없음",                       lambda r: r.scene_cuts == 0,                       4),
]
DEFAULT_MULT = 2
OUT = "./outputs/aug_weights_custom.csv"

ft = build_feature_table(train_df)
make_aug_weights(ft, [(fn, m) for _, fn, m in RULES], DEFAULT_MULT, OUT)
total = pd.read_csv(OUT).aug_mult.sum()
print(f"\n예상 학습 시간: {total * 3.5 / 3600:.1f}h (3.5초/항목, 밤 기준)")
print(f"full 커맨드: python scripts/train.py --run-name exp17_custom --aug-mult {DEFAULT_MULT} "
      f"--lr 1e-4 --epochs 1 --max-hours 0 --prompt v1_list --aug-weights {OUT} --snapshot-steps 150")